## Generation of a parcel-based building inventory for New Jersey

This notebook walks through the generation of a refined building inventory
for flood risk assessment across the full state of New Jersey. The basic
premise is that data from the National Structure Inventory (NSI) is
insufficient and poorly quality-controlled. We therefore use supplementary
data — building footprints and parcel tax-assessor records — to
dramatically improve it.

While this is not as complete or accurate as some commercial inventories,
it uses no proprietary data (e.g. First Street, CoreLogic, the now-dead
ZTRAX, Regrid), making it a reproducible example of how the NSI can be
refined for larger regions where *minimal but sufficient* alternative data
is available. The goal is to balance improvements in quality with
transparency in production — something that is not always evident in
papers relying on commercial datasets.

**Note:** The data is designed to assess *structure* losses only and does
not account for content value. NSI does define a mapping from structure
value to estimated content value (see section 6.1 of the
[Hazus Inventory Technical Manual](https://www.fema.gov/sites/default/files/documents/fema_hazus-6-inventory-technical-manual.pdf))
which could be implemented later if needed.

### Datasets
* **2022 National Structure Inventory**: U.S. Army Corps of Engineers (2022). NSI Base Data. https://nsi.sec.usace.army.mil/
* **OpenBuildingMap footprints**: Oostwegel et al. (2025). From Footprints to Functions. *Scientific Data*, 12(1), 1699.
* **NJ MOD-IV Tax Assessor and Merged Parcel Data**: ArcGIS (2026). https://pumagic.maps.arcgis.com/home/item.html?id=406cf6860390467d9f328ed19daa359d

### Key references
* Pollack et al. (2025). Unrefined national building inventories can mislead risk assessments. SSRN 5575271.
* Pollack, Doss-Gollin, Srikrishnan & Keller (2025). UNSAFE framework. *JOSS*, 10(115), 7527.
* Kijewski-Correa et al. (2023). Validation of augmented parcel approach. *Natural Hazards Review*, 24(3).
* Kijewski-Correa et al. (2020). Geospatial environments for hurricane risk assessment in NJ. *Frontiers in Built Environment*, 6.
* Zsarnóczay et al. (2025). Open-source simulation platform for natural hazards engineering. *Frontiers in Built Environment*, 11.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import geopandas as gpd

import parsers

### Configuration

All paths and runtime flags are defined here so they are easy to find
and modify without hunting through the notebook.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE_PATH   = "/scratch/gpfs/GVILLARI/lt0663/risk_analysis"
DATA_PATH   = f"{BASE_PATH}/nj_parcels"
OUT_PATH    = f"{DATA_PATH}/nj_inventory"

NSI_PATH      = f"{DATA_PATH}/nsi_new_jersey.gpkg"
NJ_PARCEL_PATH = f"{DATA_PATH}/parcels_MOD4_Statewide.gdb/Statewide_Parcels_MODIV.gdb"
NJ_SHP_PATH   = f"{DATA_PATH}/NJ_SHP/NJ_State_Boundary.shp"
OBM_PATH      = f"{BASE_PATH}/data/OpenBuildingMap/building.032010.gpkg"
OBM_NJ_PATH   = f"{DATA_PATH}/obm_nj.gpkg"
DDF_PATH      = f"{BASE_PATH}/raritan_parcels/hazus_mapping_raritan_enh.csv"

# ── Runtime flags ────────────────────────────────────────────────────────────
# Set CLEAN_OBM=True on first run to clip and save the NJ footprint subset.
# Subsequent runs can leave it False to load the cached file.
CLEAN_OBM = False

# ── Footprint-relocation parameters ──────────────────────────────────────────
# If a secondary footprint area / primary footprint area > this threshold,
# treat all footprints in the parcel as similar-sized and split values equally.
SIMILAR_SIZE_RATIO = 0.75

# Filter footprints smaller than the 1st-percentile area (sheds, garages, etc.).
# Assigned after OBM data is loaded; placeholder here for documentation.
MIN_BUILDING_SQM = None   # set after loading nj_buildings

# Minimum overlap fraction (footprint area inside parcel / total footprint area)
# to count as a valid intersection. Filters marginal boundary overlaps.
MIN_OVERLAP_RATIO = 0.075

---

## Part 1 — Reading and Cleaning Data

Standard read-in and column selection. The OBM footprints were already
processed on the HPC; the `CLEAN_OBM` flag controls whether to re-clip
them or load the cached result.

In [ ]:
nsi_nj     = gpd.read_file(NSI_PATH)
parcels_nj = gpd.read_file(NJ_PARCEL_PATH, layer='Cad_parcel_mod4')

Inspect all available fields — most will be dropped later.

In [ ]:
print("NSI columns:",     list(nsi_nj.columns))
print("Parcel columns:",  list(parcels_nj.columns))

### Additional lots

The first processing issue is the presence of "additional lots" in the
parcel data. These link a parcel to another that shares the same
characteristics or is part of the same property. Unfortunately, many
assessors also use the `ADD_LOTS1` and `ADD_LOTS2` fields as random
overflow for other data, creating ambiguity:

In [ ]:
parcels_nj.ADD_LOTS1.value_counts()

We accept lots as connected when `ADD_LOTS` contains either a single
number, a comma-separated list of lot numbers, or a PAMS_PIN. The merge
unions child geometries into their parent and drops the child rows.
Some data will be lost for non-standard entries, but correctly filled
lots (e.g. large apartment complexes) will benefit.

In [ ]:
%%time
parcels_nj = parsers.merge_additional_lots(parcels_nj, debug_path=None)

### Column selection

Keep only fields that are useful for occupancy classification, value
estimation, or spatial joining. Strip the HAZUS sub-type suffix from
NSI `occtype` (e.g. `RES1-2SWB` → `RES1`) because stories and
foundation type are already encoded in separate columns.

In [ ]:
COLS_PARCELS = [
    'PAMS_PIN', 'PROP_CLASS', 'COUNTY', 'PROP_LOC', 'GIS_PIN',
    'LAND_VAL', 'IMPRVT_VAL', 'NET_VALUE', 'BLDG_DESC', 'LAND_DESC',
    'PROP_USE', 'BLDG_CLASS', 'DWELL', 'COMM_DWELL',
    'Shape_Length', 'Shape_Area', 'geometry',
]

COLS_NSI = [
    'fd_id', 'cbfips', 'st_damcat', 'occtype', 'bldgtype',
    'num_story', 'found_type', 'found_ht', 'grnd_elv_m', 'ground_elv',
    'geometry', 'val_struct', 'val_cont',
]

nsi_nj_clean     = nsi_nj[COLS_NSI].copy()
parcels_nj_clean = parcels_nj[COLS_PARCELS].copy()

nsi_nj_clean["occtype"] = nsi_nj_clean["occtype"].str.replace(
    r"-.*$", "", regex=True
)
nsi_nj_clean.head()

### OBM footprints

Footprints improve building location estimates, especially for large
apartment complexes, nursing homes, and townhouses where the parcel
centroid is not representative. We read them now but do not relocate
points until all parcel and NSI processing is complete.

The main limitation is that footprints alone cannot supply parcel
attributes (value, occupancy type); they are used for position only.
Any remaining errors in value or type propagate from the parcel data.

In [ ]:
%%time
if CLEAN_OBM:
    obm_tile = gpd.read_file(OBM_PATH)
    nj_mask  = gpd.read_file(NJ_SHP_PATH)

    if obm_tile.crs != nj_mask.crs:
        obm_tile = obm_tile.to_crs(nj_mask.crs)

    obm_tile["geometry"] = obm_tile.geometry.make_valid()
    nj_mask["geometry"]  = nj_mask.geometry.make_valid()
    print("Geometries validated")

    # Use spatial index + bounding-box pre-filter before full clip
    obm_tile.sindex
    nj_mask.sindex
    print("Clipping to NJ boundary...")
    bounds = nj_mask.total_bounds
    obm_filtered = obm_tile.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]
    nj_buildings = obm_filtered.clip(nj_mask)
    nj_buildings.to_file(OBM_NJ_PATH)
    print(f"Saved clipped footprints to {OBM_NJ_PATH}")

else:
    nj_buildings = gpd.read_file(OBM_NJ_PATH)

nj_buildings

---

## Part 2 — Monte Carlo Simulation for First Floor Elevation

Parcel data rarely includes foundation heights or basement information
as standard fields (though some assessors encode them in `BLDG_DESC`,
which we parse later). Following the UNSAFE framework, we resample
foundation heights from NSI using triangular distributions derived from
a FATHOM/USACE survey.

We produce quantiles rather than a single value so the user can choose
how conservative to be. The "minimal" output defaults to the median
(q50). Foundation *type* is not resampled from tract-level distributions
(as in the Philly study) because we do not have independent tract-level
data to draw from beyond NSI.

In [ ]:
print(f"NSI foundation types: {nsi_nj_clean.found_type.unique()}")
print(f"Crawl space count (C): {(nsi_nj_clean.found_type == 'C').sum():,}")

In [ ]:
%%time
nsi_nj_clean = parsers.resample_ffe(nsi_nj_clean)
nsi_nj_clean.head()

---

## Part 3 — Parse the Parcels

Using the NHERI rule sets and NJ Tax Appraisal Manual, we extract
occupancy type, story count, foundation type, and basement presence
from the MOD-IV fields. All parsing logic lives in `parsers.py`.

Characteristics sought:
- **Occupancy type** (HAZUS code) — from PROP_CLASS, PROP_USE, BLDG_CLASS,
  BLDG_DESC, and DWELL in priority order
- **Foundation type / basement** — from BLDG_DESC regex
- **Stories** — from BLDG_DESC regex; falls back to architectural style

In [ ]:
parcels_nj_clean.head()

In [ ]:
parcels_nj_clean['PARSED_OCCTYPE'] = parcels_nj_clean.apply(
    parsers.get_hazus_occupancy, axis=1
)

In [ ]:
parcels_nj_parsed = parsers.add_parsed_bldg_columns(parcels_nj_clean)
parcels_nj_parsed.head()

### Parse statistics

How much of the parcel data could we extract occupancy, foundation, and
story information from?

In [ ]:
total = len(parcels_nj_clean)
missing_prop     = parcels_nj_clean["PROP_CLASS"].isna()
missing_inferred = parcels_nj_clean["PARSED_OCCTYPE"].isna()
both_missing     = missing_prop & missing_inferred
inferred_only    = ~missing_prop & missing_inferred

print(f"Total parcels: {total:,}\n")
print(f"Missing PROP_CLASS:      {missing_prop.sum():,} ({missing_prop.mean()*100:.2f}%)")
print(f"Missing PARSED_OCCTYPE:  {missing_inferred.sum():,} ({missing_inferred.mean()*100:.2f}%)")
print(f"Both missing:            {both_missing.sum():,} ({both_missing.mean()*100:.2f}%)")
print(f"Have PROP_CLASS but no inferred occtype: {inferred_only.sum():,} ({inferred_only.mean()*100:.2f}%)")

Any parcel where we could not infer an occupancy type despite having a
PROP_CLASS should be vacant land — verify this before proceeding:

In [ ]:
subset_missing = parcels_nj_clean.loc[inferred_only]
print(f"Improvement values for unresolved parcels: {subset_missing.IMPRVT_VAL.unique()}")
print(f"PROP_CLASS values:                         {subset_missing.PROP_CLASS.unique()}")

---

## Part 4 — Combine Parcels with NSI

Spatial join NSI points into parcels, then compare and resolve
occupancy type, story count, and foundation type. Parcel data always
takes priority over NSI where available.

**Design decisions:**
- NSI points on vacant land (PROP_CLASS 1) are dropped — no damage
  curve applies and improvement values are zero.
- NSI points with no matching parcel are dropped — we rely on parcel
  improvement values rather than NSI replacement cost estimates, which
  [Cox & Sanderson](https://doi.org/10.1016/j.ijdrr.2023.103696) find
  underestimate tax-assessor values at tract level.
- Mismatch cases (parcel and NSI disagree) are not automatically
  corrected; the parcel value is used and flagged for review.

In [ ]:
# Align CRS before joining
if not nsi_nj_clean.crs.equals(parcels_nj_parsed.crs):
    parcels_nj_parsed = parcels_nj_parsed.to_crs(nsi_nj_clean.crs)

print(f"NSI CRS:    {nsi_nj_clean.crs}")
print(f"Parcel CRS: {parcels_nj_parsed.crs}")

In [ ]:
PARCEL_COLS = [
    'GIS_PIN', 'PROP_CLASS', 'BLDG_DESC', 'PARSED_OCCTYPE', 'PARSED_STORIES',
    'PARSED_FOUNDATION', 'LAND_VAL', 'IMPRVT_VAL', 'NET_VALUE', 'DWELL',
    'geometry',
]

NSI_COLS = [
    'fd_id', 'cbfips', 'occtype', 'num_story', 'ffe_q05', 'ffe_q25',
    'ffe_q50', 'ffe_q75', 'ffe_q95', 'found_type', 'found_ht',
    'val_struct', 'val_cont', 'geometry',
]

parcels = parcels_nj_parsed[PARCEL_COLS].copy()
nsi     = nsi_nj_clean[NSI_COLS].copy()

In [ ]:
def join_nsi_to_parcels(nsi_gdf, parcels_gdf):
    """
    Spatial join NSI points into parcels, preserving parcel geometry.

    Parcel geometry is retained as 'parcel_geom' for use in the
    footprint-relocation step later.

    Parameters
    ----------
    nsi_gdf : GeoDataFrame
        NSI point data.
    parcels_gdf : GeoDataFrame
        Parcel polygon data.

    Returns
    -------
    GeoDataFrame
        Left join of NSI onto parcels; NSI points outside all parcels
        have NaN parcel attributes.
    """
    nsi     = nsi_gdf.copy()
    parcels = parcels_gdf.copy()

    nsi['nsi_geom']        = nsi.geometry
    parcels['parcel_geom'] = parcels.geometry

    joined = gpd.sjoin(nsi, parcels, how='left', predicate='within')
    joined = joined.drop(columns=['index_right'], errors='ignore')

    print(f"Total NSI points:      {len(nsi):,}")
    print(f"Matched to a parcel:   {joined['GIS_PIN'].notna().sum():,}")
    print(f"No matching parcel:    {joined['GIS_PIN'].isna().sum():,}")

    return joined

In [ ]:
joined = join_nsi_to_parcels(nsi, parcels)
joined.head()

In [ ]:
# Keep only NSI points that matched a parcel
matched = joined[joined['GIS_PIN'].notna()].copy()

# Inspect NSI points on vacant lots before dropping them
vacant_nsi = matched[matched['PROP_CLASS'] == '1']
print(f"NSI points on vacant lots:        {len(vacant_nsi):,}")
print(f"Vacant-lot NSI occtypes:          {vacant_nsi['occtype'].value_counts().to_dict()}")
print(f"Max improvement value (vacant):   USD {vacant_nsi['IMPRVT_VAL'].max()}")
vacant_nsi.head()

In [ ]:
# Drop vacant land and rows with missing/empty PROP_CLASS
matched = matched[matched['PROP_CLASS'] != '1']
matched = matched[matched['PROP_CLASS'].notna()]
matched = matched[matched['PROP_CLASS'] != '']
print(f"After filtering: {len(matched):,} matched points (duplicates not yet resolved)")

In [ ]:
print("Parsed occupancy types present:", matched.PARSED_OCCTYPE.unique())

### Resolve final attributes and add quality flags

In [ ]:
matched['occtype_compare'] = matched.apply(
    lambda r: parsers.compare_occtype(r['PARSED_OCCTYPE'], r['occtype']), axis=1
)
matched['stories_compare'] = matched.apply(
    lambda r: parsers.compare_stories(r['PARSED_STORIES'], r['num_story']), axis=1
)

print("Occupancy type comparison:")
print(matched['occtype_compare'].value_counts())
print("\nStories comparison:")
print(matched['stories_compare'].value_counts())

In [ ]:
matched['final_occtype']   = matched.apply(
    lambda r: parsers.resolve_occtype(r['PARSED_OCCTYPE'], r['occtype']), axis=1
)
matched['final_stories']   = matched.apply(
    lambda r: parsers.resolve_stories(r['PARSED_STORIES'], r['num_story']), axis=1
)
matched['final_foundation'] = matched.apply(
    lambda r: parsers.resolve_foundation(r['PARSED_FOUNDATION'], r['found_type']), axis=1
)
matched['has_basement'] = matched['final_foundation'].apply(
    parsers.foundation_is_basement
)

matched.head()

---

## Part 5 — Manual Corrections from Street-View Surveys

A small number of assessors entered data in non-standard ways that
cause the regex parser to misread story counts (e.g. `1437SF/T.H.`
parses as 1437 stories; `21SFR2G` parses as 21 stories because the
assessor prepended the PROP_CLASS digit). These are corrected here
rather than complicating the parser with increasingly narrow edge cases.

In [ ]:
with np.printoptions(suppress=True, precision=4):
    print(np.sort(matched[matched.PROP_CLASS == '2'].PARSED_STORIES.unique()))

**15-story residential:** Almost all are 1.5-story properties
misclassified because the assessor omitted the decimal point.
Street-view checks confirm this.

In [ ]:
matched[matched.PARSED_STORIES == 15].head()

In [ ]:
mask = (matched['PARSED_STORIES'] == 15) & (matched['PROP_CLASS'] == '2')
matched.loc[mask, 'PARSED_OCCTYPE'] = 'RES1'
matched.loc[mask, 'PARSED_STORIES'] = 1.5
print(f"Corrected {mask.sum()} rows: 15 stories → 1.5 stories (RES1)")

**10-story residential:** Non-standard entries from a single assessor
region. Classification is inconsistent with NSI and manual checks show
no pattern. Defaulted to 1 story RES1 (dominant local type).

In [ ]:
mask = (matched['PARSED_STORIES'] == 10) & (matched['PROP_CLASS'] == '2')
matched.loc[mask, 'PARSED_OCCTYPE'] = 'RES1'
matched.loc[mask, 'PARSED_STORIES'] = 1.0
print(f"Corrected {mask.sum()} rows: 10 stories → 1.0 story (RES1)")

**21-story residential:** Assessor prepended the PROP_CLASS digit (`2`)
to the real description (`1SFR2G`), making the parser read 21 stories.

In [ ]:
matched[matched.PARSED_STORIES == 21]

In [ ]:
mask = (matched['PARSED_STORIES'] == 21) & (matched['PROP_CLASS'] == '2')
matched.loc[mask, 'PARSED_OCCTYPE'] = 'RES1'
matched.loc[mask, 'PARSED_STORIES'] = 1.0
print(f"Corrected {mask.sum()} rows: 21 stories → 1.0 story (RES1)")

**>1000 stories:** Square footage entered in the building description
field. Too many to correct manually; story count falls back to NSI in
the resolve step. A MAX_STORIES catch in the regex could reduce these
further in a future iteration.

---

## Part 6 — Aggregate NSI Points to Parcels

Multiple NSI points within the same parcel are aggregated to a single
row. Values (struct, cont, land, improvement) are summed across points;
conflict flags are set where NSI points within one parcel disagree on
type, story count, or foundation.

This is also the natural split point for future parallelisation: the
`matched` dataframe can be partitioned by county and each partition
processed independently.

In [ ]:
def aggregate_nsi_to_parcel(group):
    """
    Aggregate multiple NSI points within a parcel to a single row.

    Parcel geometry is preserved as 'parcel_geom' for the footprint-
    relocation step. Values are summed; categorical attributes use the
    first entry (all points in a parcel should share parcel-level fields).

    This is also a candidate location for future outlier handling before
    national-scale runs where UQ matters more.

    Parameters
    ----------
    group : DataFrameGroupBy group
        All NSI rows matching a single GIS_PIN.

    Returns
    -------
    pd.Series
    """
    return pd.Series({
        # Parcel attributes (identical across all points in group)
        'GIS_PIN':           group['GIS_PIN'].iloc[0],
        'PROP_CLASS':        group['PROP_CLASS'].iloc[0],
        'PARSED_OCCTYPE':    group['PARSED_OCCTYPE'].iloc[0],
        'PARSED_STORIES':    group['PARSED_STORIES'].iloc[0],
        'PARSED_FOUNDATION': group['PARSED_FOUNDATION'].iloc[0],
        'LAND_VAL':          group['LAND_VAL'].iloc[0],
        'IMPRVT_VAL':        group['IMPRVT_VAL'].iloc[0],
        'NET_VALUE':         group['NET_VALUE'].iloc[0],
        'parcel_geom':       group['parcel_geom'].iloc[0],

        # NSI point counts and IDs
        'nsi_count':  len(group),
        'nsi_fd_ids': list(group['fd_id']),

        # NSI attributes
        'cbfips':       group['cbfips'].iloc[0],
        'nsi_occtype':  group['occtype'].iloc[0],
        'nsi_stories':  group['num_story'].iloc[0],
        'found_type':   group['found_type'].iloc[0],
        'nsi_val_struct': group['val_struct'].sum(),
        'nsi_val_cont':   group['val_cont'].sum(),

        # Foundation height quantiles (take minimum across points)
        'found_ht_q05': group['ffe_q05'].min(),
        'found_ht_q25': group['ffe_q25'].min(),
        'found_ht_q50': group['ffe_q50'].min(),
        'found_ht_q75': group['ffe_q75'].min(),
        'found_ht_q95': group['ffe_q95'].min(),

        # Final resolved attributes
        'final_occtype':    group['final_occtype'].iloc[0],
        'final_stories':    group['final_stories'].iloc[0],
        'final_foundation': group['final_foundation'].iloc[0],
        'has_basement':     group['has_basement'].iloc[0],

        # Data quality flags
        'occtype_compare': group['occtype_compare'].iloc[0],
        'stories_compare': group['stories_compare'].iloc[0],

        # Intra-parcel conflict flags
        'nsi_occtype_conflict':   group['occtype'].nunique() > 1,
        'nsi_stories_conflict':   group['num_story'].nunique() > 1,
        'nsi_foundation_conflict': group['found_type'].nunique() > 1,
    })

In [ ]:
%%time
parcel_agg = (
    matched
    .groupby('GIS_PIN')
    .apply(aggregate_nsi_to_parcel)
    .reset_index(drop=True)
)

print(f"Aggregated parcels:                  {len(parcel_agg):,}")
print(f"Parcels with multiple NSI points:    {(parcel_agg['nsi_count'] > 1).sum():,}")
print(f"NSI occtype conflicts:               {parcel_agg['nsi_occtype_conflict'].sum():,}")
print(f"NSI stories conflicts:               {parcel_agg['nsi_stories_conflict'].sum():,}")
print(f"NSI foundation conflicts:            {parcel_agg['nsi_foundation_conflict'].sum():,}")

parcel_agg.head()

---

## Part 7 — Reposition Buildings with OBM Footprints

Using footprint centroids instead of parcel centroids improves building
location estimates for:
- Large parcels where the building sits far from the centroid
- Row/townhouses where multiple RES3 parcels share one footprint
- Apartment complexes with multiple footprints on one lot

See [Wing et al. (2022)](https://doi.org/10.1111/jfr3.12851) for
reference on parcel-footprint matching; note we apply stricter
type-matching because ZTRAX is no longer available.

**Two key adjustments:**
1. A footprint crossing multiple RES3 parcels gets its subtype
   recalculated from the number of parcels spanned (proxy for unit count).
2. Multiple footprints on one lot have values split equally among them.

In [ ]:
# Set minimum building area threshold after data is loaded
MIN_BUILDING_SQM = nj_buildings.geometry.area.quantile(0.01)
print(f"Min building area (1st percentile): {MIN_BUILDING_SQM:.1f} m²")

In [ ]:
def relocate_buildings_with_footprints(
    parcel_agg,
    footprints_gdf,
    similar_size_ratio=0.005,
    min_building_sqm=10,
    min_overlap_ratio=0.005,
):
    """
    Relocate building points using OBM footprint centroids.

    Where a good footprint match is found the point moves to the
    footprint centroid; where not it falls back to the parcel centroid.
    This is still better than using parcel centroids uniformly, but is
    not foolproof — see the data quality section of the notebook for
    known failure modes.

    Relocation rules
    ----------------
    - ``multi_parcel_footprint``: footprint spans >1 parcel → aggregate
      parcels, adjust RES3 subtype by number of parcels spanned.
    - ``res1_largest_footprint``: RES1 with multiple differently-sized
      footprints → keep only the largest.
    - ``res1_similar_sizes``: RES1 with similarly-sized footprints →
      keep all and split values.
    - ``non_res1_multi_footprint``: non-RES1 with multiple footprints →
      keep all and split values.
    - ``single_footprint``: one footprint per parcel → straightforward
      relocation.
    - ``parcel_centroid``: no footprint found → fall back.

    Parameters
    ----------
    parcel_agg : GeoDataFrame
        Aggregated parcel data with 'parcel_geom' column.
    footprints_gdf : GeoDataFrame
        OBM building footprints.
    similar_size_ratio : float
        Secondary/primary area ratio above which footprints are treated
        as similar size. Default 0.005.
    min_building_sqm : float
        Minimum footprint area in m². Default 10.
    min_overlap_ratio : float
        Minimum (footprint area inside parcel / total footprint area)
        to count as a valid intersection. Default 0.005.

    Returns
    -------
    GeoDataFrame
        One row per building with relocated point geometry.
    """
    parcels = parcel_agg.copy()
    if 'parcel_geom' not in parcels.columns:
        raise ValueError("parcel_agg must have a 'parcel_geom' column")

    parcels = gpd.GeoDataFrame(parcels, geometry='parcel_geom',
                               crs=footprints_gdf.crs)

    footprints = footprints_gdf.copy()
    footprints['fp_area']     = footprints.geometry.area
    footprints = footprints[footprints['fp_area'] >= min_building_sqm].copy()
    footprints['fp_id']       = footprints['id']
    footprints['fp_centroid'] = footprints.geometry.centroid

    # Join footprints to parcels
    fp_to_parcels = gpd.sjoin(
        footprints,
        parcels[['GIS_PIN', 'parcel_geom']].set_geometry('parcel_geom'),
        how='inner',
        predicate='intersects',
    ).drop(columns=['index_right'], errors='ignore')

    fp_to_parcels = fp_to_parcels.merge(
        parcels[['GIS_PIN', 'parcel_geom']],
        on='GIS_PIN', how='left', suffixes=('', '_parcel'),
    )

    fp_to_parcels['fp_parcel_count'] = fp_to_parcels.groupby('fp_id')['GIS_PIN'].transform('nunique')
    fp_to_parcels['parcel_fp_count'] = fp_to_parcels.groupby('GIS_PIN')['fp_id'].transform('nunique')
    fp_to_parcels['fp_rank']         = fp_to_parcels.groupby('GIS_PIN')['fp_area'].rank(ascending=False)
    fp_to_parcels['max_fp_area']     = fp_to_parcels.groupby('GIS_PIN')['fp_area'].transform('max')
    fp_to_parcels['area_ratio']      = fp_to_parcels['fp_area'] / fp_to_parcels['max_fp_area']

    fp_to_parcels = fp_to_parcels.merge(
        parcels.drop(columns=['parcel_geom']), on='GIS_PIN', how='left'
    )

    # Classify each footprint-parcel pair into a relocation rule
    rule3_mask = fp_to_parcels['fp_parcel_count'] > 1

    rule2a_mask = (
        ~rule3_mask
        & (fp_to_parcels['final_occtype'] == 'RES1')
        & (fp_to_parcels['parcel_fp_count'] > 1)
        & (fp_to_parcels['fp_rank'] == 1)
        & (fp_to_parcels.groupby('GIS_PIN')['area_ratio'].transform(
            lambda x: x.iloc[1] if len(x) > 1 else 0) < similar_size_ratio)
    )

    rule2b_mask = (
        ~rule3_mask
        & (fp_to_parcels['final_occtype'] == 'RES1')
        & (fp_to_parcels['parcel_fp_count'] > 1)
        & ~rule2a_mask
        & (fp_to_parcels.groupby('GIS_PIN')['area_ratio'].transform(
            lambda x: x.iloc[1] if len(x) > 1 else 0) >= similar_size_ratio)
    )

    rule1_mask = (
        ~rule3_mask
        & (fp_to_parcels['final_occtype'] != 'RES1')
        & (fp_to_parcels['parcel_fp_count'] > 1)
    )

    rule4_mask = ~rule3_mask & (fp_to_parcels['parcel_fp_count'] == 1)

    fp_to_parcels['rule'] = None
    fp_to_parcels.loc[rule3_mask,  'rule'] = 'multi_parcel_footprint'
    fp_to_parcels.loc[rule2a_mask, 'rule'] = 'res1_largest_footprint'
    fp_to_parcels.loc[rule2b_mask, 'rule'] = 'res1_similar_sizes'
    fp_to_parcels.loc[rule1_mask,  'rule'] = 'non_res1_multi_footprint'
    fp_to_parcels.loc[rule4_mask,  'rule'] = 'single_footprint'

    fp_to_parcels = fp_to_parcels[fp_to_parcels['rule'].notna()].copy()

    def adjust_res3_occtype(occtype, n_parcels):
        """Adjust RES3 subtype based on number of parcels a footprint spans."""
        if not isinstance(occtype, str) or not occtype.startswith('RES3'):
            return occtype
        unit_ranges = {
            'RES3A': (2, 2),  'RES3B': (3, 4),  'RES3C': (5, 9),
            'RES3D': (10, 19), 'RES3E': (20, 49), 'RES3F': (50, 999),
        }
        for occ, (low, high) in unit_ranges.items():
            if low <= n_parcels <= high:
                return occ
        return 'RES3F' if n_parcels >= 50 else occtype

    output_rows = []
    processed_parcels = set()

    # Rule 3: aggregate parcels per footprint
    for fp_id, group in fp_to_parcels[fp_to_parcels['rule'] == 'multi_parcel_footprint'].groupby('fp_id'):
        n_parcels       = group['GIS_PIN'].nunique()
        base_occtype    = group['final_occtype'].mode().iloc[0]
        adjusted_occtype = adjust_res3_occtype(base_occtype, n_parcels)
        unique_parcels  = group.drop_duplicates('GIS_PIN')

        output_rows.append({
            'GIS_PIN':          ','.join(group['GIS_PIN'].unique().astype(str)),
            'nsi_val_struct':   unique_parcels['nsi_val_struct'].sum(),
            'nsi_val_cont':     unique_parcels['nsi_val_cont'].sum(),
            'LAND_VAL':         unique_parcels['LAND_VAL'].sum(),
            'IMPRVT_VAL':       unique_parcels['IMPRVT_VAL'].sum(),
            'NET_VALUE':        unique_parcels['NET_VALUE'].sum(),
            'final_occtype':    adjusted_occtype,
            'original_occtype': base_occtype,
            'parcels_spanned':  n_parcels,
            'final_stories':    group['final_stories'].median(),
            'final_foundation': group['final_foundation'].mode().iloc[0] if len(group['final_foundation'].mode()) > 0 else None,
            'has_basement':     group['has_basement'].mode().iloc[0] if len(group['has_basement'].mode()) > 0 else None,
            'found_ht_q05':     group['found_ht_q05'].min(),
            'found_ht_q25':     group['found_ht_q25'].min(),
            'found_ht_q50':     group['found_ht_q50'].min(),
            'found_ht_q75':     group['found_ht_q75'].min(),
            'found_ht_q95':     group['found_ht_q95'].min(),
            'nsi_count':        unique_parcels['nsi_count'].sum(),
            'fp_id':            fp_id,
            'relocation_rule':  'multi_parcel_footprint',
            'geometry':         group['fp_centroid'].iloc[0],
        })
        processed_parcels.update(group['GIS_PIN'].unique())

    # Rules 1, 2a, 2b, 4: one row per footprint
    other_rules = fp_to_parcels[
        (fp_to_parcels['rule'] != 'multi_parcel_footprint')
        & (~fp_to_parcels['GIS_PIN'].isin(processed_parcels))
    ].copy()

    other_rules['n_fps_kept']  = other_rules.groupby('GIS_PIN')['fp_id'].transform('count')
    other_rules['val_divisor'] = np.where(other_rules['n_fps_kept'] > 1,
                                          other_rules['n_fps_kept'], 1)

    for _, row in other_rules.iterrows():
        d = row['val_divisor']
        output_rows.append({
            'GIS_PIN':          row['GIS_PIN'],
            'nsi_val_struct':   row['nsi_val_struct'] / d,
            'nsi_val_cont':     row['nsi_val_cont']   / d,
            'LAND_VAL':         row['LAND_VAL']  / d,
            'IMPRVT_VAL':       row['IMPRVT_VAL'] / d,
            'NET_VALUE':        row['NET_VALUE']  / d,
            'final_occtype':    row['final_occtype'],
            'original_occtype': row['final_occtype'],
            'parcels_spanned':  1,
            'final_stories':    row['final_stories'],
            'final_foundation': row['final_foundation'],
            'has_basement':     row['has_basement'],
            'found_ht_q05':     row['found_ht_q05'],
            'found_ht_q25':     row['found_ht_q25'],
            'found_ht_q50':     row['found_ht_q50'],
            'found_ht_q75':     row['found_ht_q75'],
            'found_ht_q95':     row['found_ht_q95'],
            'nsi_count':        row['nsi_count'] / d,
            'fp_id':            row['fp_id'],
            'relocation_rule':  row['rule'],
            'geometry':         row['fp_centroid'],
        })
        processed_parcels.add(row['GIS_PIN'])

    # Fallback: parcels with no matching footprint → parcel centroid
    for _, row in parcels[~parcels['GIS_PIN'].isin(processed_parcels)].iterrows():
        output_rows.append({
            'GIS_PIN':          row['GIS_PIN'],
            'nsi_val_struct':   row['nsi_val_struct'],
            'nsi_val_cont':     row['nsi_val_cont'],
            'LAND_VAL':         row['LAND_VAL'],
            'IMPRVT_VAL':       row['IMPRVT_VAL'],
            'NET_VALUE':        row['NET_VALUE'],
            'final_occtype':    row['final_occtype'],
            'original_occtype': row['final_occtype'],
            'parcels_spanned':  1,
            'final_stories':    row['final_stories'],
            'final_foundation': row['final_foundation'],
            'has_basement':     row['has_basement'],
            'found_ht_q05':     row['found_ht_q05'],
            'found_ht_q25':     row['found_ht_q25'],
            'found_ht_q50':     row['found_ht_q50'],
            'found_ht_q75':     row['found_ht_q75'],
            'found_ht_q95':     row['found_ht_q95'],
            'nsi_count':        row['nsi_count'],
            'fp_id':            None,
            'relocation_rule':  'parcel_centroid',
            'geometry':         row['parcel_geom'].centroid,
        })

    output_gdf = gpd.GeoDataFrame(output_rows, crs=footprints_gdf.crs)

    print("\n" + "=" * 50)
    print("BUILDING RELOCATION SUMMARY")
    print("=" * 50)
    print(f"Input parcels:   {len(parcels):,}")
    print(f"Output buildings:{len(output_gdf):,}")
    print(f"\nParameters: similar_size_ratio={similar_size_ratio}, "
          f"min_building_sqm={min_building_sqm:.1f}")
    print(f"\nBy relocation rule:")
    print(output_gdf['relocation_rule'].value_counts())

    res3_adjusted = output_gdf[
        output_gdf['final_occtype'] != output_gdf['original_occtype']
    ]
    if len(res3_adjusted) > 0:
        print(f"\nRES3 occupancy adjustments: {len(res3_adjusted):,}")
        print(res3_adjusted.groupby(['original_occtype', 'final_occtype']).size())

    print("=" * 50)
    return output_gdf

In [ ]:
%%time
final_buildings = relocate_buildings_with_footprints(
    parcel_agg,
    nj_buildings,
    similar_size_ratio=SIMILAR_SIZE_RATIO,
    min_building_sqm=MIN_BUILDING_SQM,
    min_overlap_ratio=MIN_OVERLAP_RATIO,
)

print(final_buildings.relocation_rule.value_counts())
final_buildings.head()

---

## Part 8 — Assign Depth-Damage Functions

Assign HAZUS average DDFs from the lookup table Alexander calculated
for the CONUS risk study. A single deterministic DDF per building is
used here for simplicity; the FFE quantiles allow conservative/median/
optimistic scenarios to be selected downstream.

Note: 1.5-story DDFs do not distinguish basement/no-basement in the
current lookup table. The USACE Wilmington curve without foundation
specification is used for these.

In [ ]:
ddf_lookup = pd.read_csv(DDF_PATH)
ddf_lookup.head()

In [ ]:
%%time
final_buildings_ddfs = parsers.assign_ddf_ids(final_buildings, ddf_lookup)
final_buildings_ddfs.head()

In [ ]:
print("Top DDF IDs (structure type frequency):")
print(final_buildings_ddfs.structure_ddf_id.value_counts())

---

## Part 9 — Write Outputs

### Output column reference

| Column | Description |
|---|---|
| `GIS_PIN` | Comma-separated list of parcel GIS_PINs aggregated into this building. Multiple values indicate a footprint spanning several tax parcels. |
| `LAND_VAL` | Assessed land value (excludes structures). Not directly used in risk assessment. |
| `IMPRVT_VAL` | Assessed improvement value (depreciates). Analogous to `val_struct` in NSI — use this for structure loss calculations rather than NSI replacement cost estimates. |
| `NET_VALUE` | `LAND_VAL` + `IMPRVT_VAL`. |
| `final_occtype` | HAZUS occupancy type resolved from NSI + parcel + footprint data. |
| `original_occtype` | Occupancy type before footprint-based RES3 subtype adjustment (data quality reference). |
| `parcels_spanned` | Number of tax parcels aggregated into this building record. |
| `final_stories` | Story count resolved from parcel then NSI. |
| `final_foundation` | Foundation type code resolved from parcel then NSI. |
| `has_basement` | Boolean derived from `final_foundation`. |
| `found_ht_qX` | Foundation height quantiles from MC simulation. The minimal output uses only q50; adjust for conservative/optimistic scenarios. |
| `nsi_count` | NSI points aggregated into this record. |
| `fp_id` | OBM footprint ID (None where parcel centroid was used). |
| `relocation_rule` | Which rule governed point placement. |
| `structure_ddf_id` | HAZUS Structure Function ID for loss calculation. |
| `geometry` | Point geometry of the final building location. |

In [ ]:
MINIMAL_COLS = [
    'GIS_PIN', 'LAND_VAL', 'IMPRVT_VAL', 'found_ht_q50',
    'final_occtype', 'final_stories', 'final_foundation',
    'structure_ddf_id', 'geometry',
]

if final_buildings_ddfs.crs is None:
    final_buildings_ddfs = final_buildings_ddfs.set_crs("EPSG:6347")

os.makedirs(OUT_PATH, exist_ok=True)

# Full dataset including quality-control flags
full_path    = f"{OUT_PATH}/nj_inventory_full_w_obm_adj.gpkg"
minimal_path = f"{OUT_PATH}/nj_inventory_minimal_w_obm_adj.gpkg"

final_buildings_ddfs.to_file(full_path)
final_buildings_ddfs[MINIMAL_COLS].to_file(minimal_path)

print(f"Full inventory:    {full_path}")
print(f"Minimal inventory: {minimal_path}")

---

## Open Questions

1. **Is improvement value comparable to NSI replacement cost?** Cox &
   Sanderson (2023) find NSI underestimates tax-assessor values at tract
   level in the Cascadia earthquake context. How should we frame this for
   flood risk?

2. **Temporal mismatch:** This data represents the 2024 tax year; NSI is
   from 2022. Using a
   [CPI inflation calculator](https://www.bls.gov/data/inflation_calculator.htm),
   USD 1 in September 2021 ≈ USD 1.06 in 2024. Should we apply a CPI
   adjustment or a property-specific index?

3. **Uncertainty acknowledgement:** How do we communicate the uncertainty
   from the assumption chain (parcel priority, regex parsing, FFE MC) in
   a way that is honest but not paralyzing? ±30% std dev on DDF +
   FFE quantiles as a starting point?

4. **Content value:** Pollack et al. (2025) use structure value only
   (see [their notebook](https://github.com/abpoll/nsi_fit/blob/main/notebooks/prepare_data.ipynb)).
   Is that acceptable here, or should we implement the HAZUS content
   scaling (occtype + sqft + depreciation factor)?

5. **Mobile homes (RES2):** Systematically undercounted. A Stephen
   Strader-style tornado study approach could help. Worth a dedicated
   follow-up given the social vulnerability implications.